# Interview Copilot Colab Training

Bu notebook, `sft.v2` dataset ve yeni dual-track LoRA presetleriyle Colab uzerinde adapter egitimi baslatmak icin hazirlandi.

Audience:
- Interview Copilot training pipeline'ini Colab'a tasimak isteyen gelistirici.

Prerequisites:
- Google Drive mount yetkisi
- Hugging Face token (gerekliyse)
- Repo veya `artifacts/train_bundle` klasoru Drive'a kopyalanmis olmali

Learning goals:
- Bundle'i Drive'dan yuklemek
- Stable veya aggressive profile secmek
- Adapter ve raporlari tekrar Drive'a yazmak


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi


## Repo veya Bundle Yolunu Sec
`PROJECT_ROOT` dogrudan repo klasoru olabilir veya `train_bundle` kopyasini iceren bir klasor olabilir.


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path('/content/drive/MyDrive/LiveTranslate')
BUNDLE_DIR = PROJECT_ROOT / 'artifacts' / 'train_bundle'
assert PROJECT_ROOT.exists(), PROJECT_ROOT
os.chdir(PROJECT_ROOT)
print(PROJECT_ROOT)
print(BUNDLE_DIR if BUNDLE_DIR.exists() else 'bundle_not_found')


In [ ]:
!python -V
!pip install -U pip setuptools wheel
!pip install -U accelerate datasets peft sentencepiece safetensors transformers trl bitsandbytes pyyaml
!pip install -U axolotl


## Profil Secimi
- `PROFILE_FAMILY='llama8b'` varsayilan kalite adayi
- `PROFILE_TIER='aggressive'` Colab full train icin
- `PROFILE_TIER='stable'` daha guvenli bellek kullanimi icin


In [ ]:
PROFILE_FAMILY = 'llama8b'
PROFILE_TIER = 'aggressive'
DATASET = PROJECT_ROOT / 'artifacts' / 'finetune' / 'interview_train.sft.v2.jsonl'
VALID_DATASET = PROJECT_ROOT / 'artifacts' / 'finetune' / 'interview_train.sft.v2.valid.jsonl'
QUALITY_REPORT = PROJECT_ROOT / 'artifacts' / 'finetune' / 'interview_quality_report.v2.json'
OUT_DIR = PROJECT_ROOT / 'artifacts' / 'finetune' / 'lora_interview_colab'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(DATASET.exists(), VALID_DATASET.exists(), QUALITY_REPORT.exists())


In [ ]:
!python scripts/train_lora_interview.py --dataset {DATASET} --valid-dataset {VALID_DATASET} --out-dir {OUT_DIR} --profile-family {PROFILE_FAMILY} --profile-tier {PROFILE_TIER} --train-env colab --quality-report {QUALITY_REPORT} --require-quality-pass


## Full Train Baslat
Bir onceki hucre config ve komut dosyalarini uretir. A?a??daki hucre smoke veya full train baslatir.


In [ ]:
!python scripts/train_lora_interview.py --dataset {DATASET} --valid-dataset {VALID_DATASET} --out-dir {OUT_DIR} --profile-family {PROFILE_FAMILY} --profile-tier {PROFILE_TIER} --train-env colab --quality-report {QUALITY_REPORT} --require-quality-pass --run-full


In [ ]:
!python scripts/export_train_bundle.py --dataset {DATASET} --valid-dataset {VALID_DATASET} --quality-report {QUALITY_REPORT} --judge-config configs/judge_config.json --output-dir artifacts/train_bundle
!ls -lah artifacts/train_bundle
